In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path
plt.style.use("dark_background")
%matplotlib inline

In [ ]:
# Add parent directory for pyquant imports
import sys
sys.path.insert(0, '..')
from pyquant.torch_spline import CubicSpline1D
from pyquant import heston_sim

In [ ]:
# Load data
data_dir = Path('../../../data')
fwd_ois = pd.read_csv(data_dir / 'forward_ois.csv')
fwd_key_rate = pd.read_csv(data_dir / 'forward_key_rate.csv')
vol_key_rate = pd.read_csv(data_dir / 'volatility_key_rate.csv')

print("Volatility surface maturities (years):")
print(sorted(vol_key_rate['time_to_maturity'].unique()))

## Time-Varying Theta Parameters

We define 13 theta values at key maturities to capture the volatility term structure.

In [ ]:
# Key maturities for theta interpolation (exclude very short < 0.25y)
THETA_MATURITIES = torch.tensor([0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
N_THETA = len(THETA_MATURITIES)

print(f"Number of theta parameters: {N_THETA}")
print(f"Theta maturities (years): {THETA_MATURITIES.tolist()}")
print(f"Total model parameters: 4 (v0, κ, ε, λ) + {N_THETA} (θ₁...θ₁₃) + 2 (γ, ξ) = {4 + N_THETA + 2}")

## Enhanced CIR Generator with Time-Varying θ(t)

Linear interpolation of theta values across the timeline.

In [ ]:
def generate_cir_time_varying_theta(n_paths, timeline, v0, kappa, theta_values, epsilon, floor_variance):
    """
    Generate CIR paths with time-varying theta.
    
    Args:
        n_paths: Number of Monte Carlo paths
        timeline: Time grid
        v0: Initial variance
        kappa: Mean reversion speed
        theta_values: Tensor of theta values at THETA_MATURITIES
        epsilon: Volatility of variance  
        floor_variance: Minimum variance (for numerical stability)
    
    Returns:
        CIR paths (n_paths x len(timeline))
    """
    # Interpolate theta values to timeline
    theta_t = torch.zeros_like(timeline)
    for i, t in enumerate(timeline):
        if t <= THETA_MATURITIES[0]:
            theta_t[i] = theta_values[0]
        elif t >= THETA_MATURITIES[-1]:
            theta_t[i] = theta_values[-1]
        else:
            # Linear interpolation
            idx = torch.searchsorted(THETA_MATURITIES, t)
            t_left = THETA_MATURITIES[idx-1]
            t_right = THETA_MATURITIES[idx]
            theta_left = theta_values[idx-1]
            theta_right = theta_values[idx]
            
            weight = (t - t_left) / (t_right - t_left)
            theta_t[i] = theta_left + weight * (theta_right - theta_left)
    
    # Simulate CIR with time-varying theta
    dt = timeline.diff()
    n_steps = len(timeline) - 1
    
    v_paths = torch.zeros(n_paths, len(timeline))
    v_paths[:, 0] = v0
    
    for i in range(n_steps):
        theta_current = theta_t[i]
        dt_current = dt[i]
        
        v_current = torch.maximum(v_paths[:, i], torch.tensor(floor_variance))
        
        dW = torch.randn(n_paths) * torch.sqrt(dt_current)
        dv = kappa * (theta_current - v_current) * dt_current + epsilon * torch.sqrt(v_current) * dW
        
        v_paths[:, i+1] = torch.maximum(v_current + dv, torch.tensor(floor_variance))
    
    return v_paths

print("✅ Enhanced CIR generator with time-varying θ(t) defined")

## Initialize Enhanced Model Parameters

Structure: [v0, kappa, theta_1, ..., theta_13, epsilon, lambda, gamma, xi]

In [ ]:
# Initialize enhanced model parameters (19 total)
theta_init = 1e-4  # Initial guess for all thetas

model_params_enhanced = torch.tensor([
    1e-4,      # v0 - initial variance
    0.5,       # kappa - mean reversion speed
    # 13 theta values at key maturities
    theta_init, theta_init, theta_init, theta_init,  # 0.25, 0.5, 0.75, 1y
    theta_init, theta_init, theta_init,              # 2, 3, 4y
    theta_init, theta_init, theta_init,              # 5, 6, 7y
    theta_init, theta_init, theta_init,              # 8, 9, 10y
    1e-5,      # epsilon - vol of variance
    0.1,       # lambda - HW mean reversion
    0.1,       # gamma - key rate mean reversion
    1e-5,      # xi - key rate volatility
], requires_grad=True)

print(f"Enhanced model parameters initialized:")
print(f"  Shape: {model_params_enhanced.shape}")
print(f"  v0 = {model_params_enhanced[0].item():.6f}")
print(f"  κ = {model_params_enhanced[1].item():.6f}")
print(f"  θ₁...θ₁₃ = {model_params_enhanced[2:15].tolist()}")
print(f"  ε = {model_params_enhanced[15].item():.6f}")
print(f"  λ = {model_params_enhanced[16].item():.6f}")
print(f"  γ = {model_params_enhanced[17].item():.6f}")
print(f"  ξ = {model_params_enhanced[18].item():.6f}")

## Test Theta Interpolation

Visualize how θ(t) varies across time.

In [ ]:
# Create test timeline
timeline_test = torch.linspace(0, 10, 3651)

# Extract theta values and create interpolated curve
theta_values_test = model_params_enhanced[2:15]

# Interpolate
theta_t_test = torch.zeros_like(timeline_test)
for i, t in enumerate(timeline_test):
    if t <= THETA_MATURITIES[0]:
        theta_t_test[i] = theta_values_test[0]
    elif t >= THETA_MATURITIES[-1]:
        theta_t_test[i] = theta_values_test[-1]
    else:
        idx = torch.searchsorted(THETA_MATURITIES, t)
        t_left = THETA_MATURITIES[idx-1]
        t_right = THETA_MATURITIES[idx]
        theta_left = theta_values_test[idx-1]
        theta_right = theta_values_test[idx]
        weight = (t - t_left) / (t_right - t_left)
        theta_t_test[i] = theta_left + weight * (theta_right - theta_left)

plt.figure(figsize=(12, 6))
plt.plot(timeline_test.detach(), theta_t_test.detach(), label='θ(t) interpolated', linewidth=2)
plt.scatter(THETA_MATURITIES, theta_values_test.detach(), color='red', s=100, zorder=5, label='Calibration points')
plt.xlabel('Time (years)')
plt.ylabel('θ(t)')
plt.title('Time-Varying Mean Reversion Level')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"✅ θ(t) can be calibrated at {N_THETA} key maturities to fit volatility term structure")

## Next Steps

1. **Modify simulate_model()** to use `generate_cir_time_varying_theta()`
2. **Update calibration function** to handle 19 parameters
3. **Set parameter bounds** for each theta value
4. **Run calibration** with daily timeline and reduced MC paths
5. **Analyze results** - check if time-varying θ(t) improves volatility surface fit